# PSYCTL로 LLM 성격 즉시 조향하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/ko/01_quickstart.ipynb)

**PSYCTL이란?** [대조적 활성화 덧셈(CAA)](https://arxiv.org/abs/2312.06681)과 [양방향 선호도 최적화(BiPO)](https://arxiv.org/abs/2406.00045)를 활용해 LLM의 성격을 조향하는 Python 툴킷입니다. 프롬프트 엔지니어링과 달리, PSYCTL은 모델의 활성화(activation)를 직접 수정하여 일관되고 측정 가능한 성격 변화를 만들어냅니다.

**이 노트북에서 다루는 내용:**
1. 사전 학습된 BiPO 스티어링 벡터를 로드하고 즉각적인 성격 변화를 확인합니다
2. 기본 응답과 다양한 강도의 조향된 응답을 비교합니다
3. 5가지 성격 벡터를 나란히 비교합니다

**요구사항:** Colab 무료 티어(T4 GPU)로 충분합니다. [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) 접근 권한이 있는 [HuggingFace 토큰](https://huggingface.co/settings/tokens)이 필요합니다.

**소요 시간:** 약 5분

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## 셋업: 모델 로딩

Llama-3.1-8B-Instruct를 4비트 양자화(약 5GB VRAM)로 로드하여 무료 Colab T4에서 실행할 수 있도록 합니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_ID} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {model.device} | Memory: {model.get_memory_footprint() / 1e9:.1f} GB")

## 스티어링 벡터 다운로드

[dalekwon/bipo-steering-vectors](https://huggingface.co/dalekwon/bipo-steering-vectors)에서 사전 학습된 **우호성(Agreeableness)** BiPO 벡터를 다운로드합니다.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

VECTOR_REPO = "dalekwon/bipo-steering-vectors"
VECTOR_DIR = Path("./vectors")
VECTOR_DIR.mkdir(exist_ok=True)

vector_path = Path(hf_hub_download(
    repo_id=VECTOR_REPO,
    filename="bipo_steering_english_agreeableness.safetensors",
    local_dir=str(VECTOR_DIR),
    token=os.environ.get("HF_TOKEN"),
))
print(f"Downloaded vector: {vector_path}")

## 기본 응답 vs 조향된 응답

동일한 프롬프트에 대해 스티어링 강도에 따라 극적으로 다른 응답이 생성됩니다. 양의 강도는 우호성을 높이고, 음의 강도는 우호성을 낮춥니다.

In [ ]:
from psyctl.core.steering_applier import SteeringApplier

applier = SteeringApplier()


def generate_baseline(prompt: str, max_new_tokens: int = 200) -> str:
    """Generate without steering."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def generate_steered(prompt: str, strength: float, vpath=vector_path, max_new_tokens: int = 200) -> str:
    """Generate with steering vector applied."""
    return applier.apply_steering(
        model=model,
        tokenizer=tokenizer,
        steering_vector_path=vpath,
        input_text=prompt,
        strength=strength,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
    )

In [ ]:
prompt = "My coworker keeps taking credit for my ideas. What should I do?"

print(f"Prompt: {prompt}")
print("=" * 70)

print("\n--- Baseline (no steering) ---")
print(generate_baseline(prompt))

print("\n--- Steered: Agreeableness +2.0 ---")
print(generate_steered(prompt, strength=2.0))

print("\n--- Steered: Agreeableness -2.0 (disagreeable) ---")
print(generate_steered(prompt, strength=-2.0))

## 강도 변화 실험

-2.0부터 +3.0까지 성격이 어떻게 점진적으로 변하는지 확인해봅니다.

In [ ]:
prompt = "Someone cuts in line at the coffee shop. How do you react?"
strengths = [-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]

print(f"Prompt: {prompt}")
print("=" * 70)

for s in strengths:
    label = "Baseline" if s == 0.0 else f"Agreeableness {s:+.1f}"
    print(f"\n--- {label} ---")
    if s == 0.0:
        print(generate_baseline(prompt, max_new_tokens=150))
    else:
        print(generate_steered(prompt, strength=s, max_new_tokens=150))

## 다양한 성격 탐색

[dalekwon/bipo-steering-vectors](https://huggingface.co/dalekwon/bipo-steering-vectors) 레포에는 5가지 영어 성격 벡터가 포함되어 있습니다:

| 벡터 | 설명 |
|------|------|
| `agreeableness` | 협조적, 신뢰하는, 동정적 |
| `neuroticism` | 감정적으로 반응적, 불안한, 예민한 |
| `awfully_sweet` | 극도로 친절하고 따뜻하며 넘치게 긍정적 |
| `paranoid` | 의심 많고, 불신하며, 과잉 경계하는 |
| `very_lascivious` | 대담하고, 감각 추구적이며, 경계를 넘는 |

In [ ]:
VECTOR_FILES = {
    "agreeableness": "bipo_steering_english_agreeableness.safetensors",
    "neuroticism": "bipo_steering_english_neuroticism.safetensors",
    "awfully_sweet": "bipo_steering_english_awfully_sweet.safetensors",
    "paranoid": "bipo_steering_english_paranoid.safetensors",
    "very_lascivious": "bipo_steering_english_very_lascivious.safetensors",
}

vector_paths = {}
for name, filename in VECTOR_FILES.items():
    vector_paths[name] = Path(hf_hub_download(
        repo_id=VECTOR_REPO, filename=filename,
        local_dir=str(VECTOR_DIR), token=os.environ.get("HF_TOKEN"),
    ))
    print(f"  {name}: ready")

print(f"\nAll {len(vector_paths)} vectors downloaded.")

In [ ]:
# Side-by-side comparison: same prompt, all personalities
prompt = "Tell me about yourself."
strength = 2.5

print(f"Prompt: \"{prompt}\" (strength={strength:+.1f})")
print("=" * 70)

print("\n[Baseline]")
print(generate_baseline(prompt, max_new_tokens=120))

for name, vpath in vector_paths.items():
    print(f"\n[{name}]")
    print(generate_steered(prompt, strength=strength, vpath=vpath, max_new_tokens=120))
    print("-" * 70)

## 다음 단계

- **[02_measure_personality.ipynb](./02_measure_personality.ipynb)** -- 표준화된 심리 검사 도구로 성격 변화를 정량화합니다
- **[04_extract_vector.ipynb](./04_extract_vector.ipynb)** -- 나만의 스티어링 벡터를 추출합니다
- **[GitHub](https://github.com/modulabs-personalab/psyctl)** -- 레포에 스타를 누르고, 이슈를 열고, 기여해주세요